# DPO stage - DPO of SFT-GPT-2 small on hh-rlhf (`harmless-base`)

Produces the anchor checkpoint for the DPO-SAE project: DPO GPT-2 small finetuned with a plain
LM objective on both rejected and chosen.

**Run order:** mount Drive, set secrets, then run the cells top to bottom for the full
~21k-pair run.

## 1. Drive

First, before anything else. Checkpoints go to Drive, not to the VM disk - a disconnect
partway through an unmounted run loses every checkpoint.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
!pip -q install -U transformers datasets accelerate wandb

## 2. Secrets and environment

The key comes from the Colab secrets panel (the key icon in the left sidebar), never from a
cell. A pasted key gets saved into the notebook's stored output; so does an interactive
`wandb.login()`. Add a secret named `WANDB_API_KEY` and give this notebook access to it.

In [ ]:
import os

from google.colab import userdata

os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ["WANDB_PROJECT"] = "DPO-SAE"     # shared with the later DPO and SAE runs
os.environ["WANDB_LOG_MODEL"] = "false"     # checkpoints belong on Drive, not in wandb
os.environ["WANDB_WATCH"] = "false"

# Optional: park the HF cache on Drive so gpt2 and harmless-base survive a fresh VM.
# Both are small, so this is a convenience, not a requirement.
# os.environ["HF_HOME"] = "/content/drive/MyDrive/DPO-SAE/hf-cache"

## 3. Configuration

Two things here are load-bearing and easy to break.

**The budget is counted in pairs, not steps or epochs.** Step count moves with batch size;
the pair budget does not. A running counter drives the stopping condition, the checkpoint
cadence and the validation cadence. `max_steps` and `save_steps` do not.

**Half the split, sampled.** hh-rlhf is not shuffled in a way that makes a prefix
representative, so the half is drawn with a seeded shuffle. The other half is reserved for
DPO: training both stages on the same rows means the DPO reference has already memorized the
`chosen` completions, which shrinks the logprob gap the DPO loss works on. `DATA_SEED` and
the row range are written to the run config and to a manifest so the DPO stage can
reconstruct exactly which half this model saw.

In [ ]:
import torch

MODEL_NAME = "gpt2"           # GPT-2 small, 124M
DATASET = "Anthropic/hh-rlhf"
DATA_DIR = "harmless-base"

DATA_SEED = 1337              # the DPO stage needs this to reconstruct the split
SFT_HALF = 0                  # 0 = first half of the shuffle is SFT's, second half is DPO's

MAX_LEN = 512                 # GPT-2's context is 1024; 512 is this run's budget
MIN_COMPLETION_TOKENS = 16    # a prompt leaving less room than this is dropped
VAL_PAIRS = 512               # fixed val slice, carved from the SFT half
MASK_PROMPT_LOSS = False      # False = plain LM loss over (prompt + chosen), logged either way

CKPT_EVERY_PAIRS = 4_000      # -> 5 checkpoints + final over ~21k pairs
PER_DEVICE_BS = 8
GRAD_ACCUM = 4                # effective batch 32
LR = 5e-5
WARMUP_RATIO = 0.03
LOG_EVERY_STEPS = 10
N_SAMPLE_GENERATIONS = 3

RUN_NAME = "sft-gpt2-hh-21k"
EPOCHS = 1

DRIVE_ROOT = "/content/drive/MyDrive/DPO-SAE"
RUN_DIR = f"{DRIVE_ROOT}/{RUN_NAME}"
CKPT_DIR = f"{RUN_DIR}/checkpoints"       # the artifacts: weights + tokenizer per 4k pairs
FINAL_DIR = f"{RUN_DIR}/final"
RESUME_DIR = f"{RUN_DIR}/_resume"         # rolling Trainer state, disconnect insurance only

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RESUME_DIR, exist_ok=True)

os.environ["WANDB_RUN_ID"] = RUN_NAME
os.environ["WANDB_RESUME"] = "allow"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[env] device={DEVICE} run={RUN_NAME}")
print(f"[env] checkpoints -> {CKPT_DIR}")

## 4. Preprocessing

`chosen` and `rejected` are complete dialogues sharing every turn except the final assistant
response, so the split point is the **last** `"\n\nAssistant:"`. Everything up to and
including the marker is the prompt; everything after is the completion.

Rows are dropped when the marker is absent, when `chosen` and `rejected` do not actually
share the resulting prefix (a handful of rows have diverging earlier turns), when the
completion is empty, or when the prompt alone eats the context budget.

Filtering runs **after** sampling, so the usable count lands slightly under the nominal 21k.
The before/after counts are printed at every stage: a sudden drop in yield means the
splitter broke.

In [ ]:
MARKER = "\n\nAssistant:"


def split_dialogue(example):
    r"""Split an hh-rlhf row into (prompt, completion) at the final turn marker.

    Rows look like:
        "\n\nHuman: how do I ...\n\nAssistant: well ...\n\nHuman: ok\n\nAssistant: sure"

    Unusable rows come back empty rather than raising, and keep_row drops them.
    `rejected` is read only to verify the shared prefix - the preference signal it
    carries belongs to DPO, not to this stage.
    """
    chosen, rejected = example["chosen"], example["rejected"]

    idx = chosen.rfind(MARKER)
    if idx == -1:
        return {"prompt": "", "completion": ""}

    prompt = chosen[: idx + len(MARKER)]

    # Guard: the two must actually share the prefix.
    if not rejected.startswith(prompt):
        return {"prompt": "", "completion": ""}

    return {"prompt": prompt, "completion": chosen[idx + len(MARKER):]}


def keep_row(example):
    return len(example["prompt"]) > 0 and len(example["completion"].strip()) > 0

In [ ]:
from datasets import load_dataset

raw = load_dataset(DATASET, data_dir=DATA_DIR, split="train")
RAW_N = len(raw)

# Seeded shuffle, then halve. Not the first 21k rows: the split is not ordered in a way
# that makes a prefix representative.
shuffled = raw.shuffle(seed=DATA_SEED)
mid = RAW_N // 2
SFT_ROWS = (0, mid) if SFT_HALF == 0 else (mid, RAW_N)
DPO_ROWS = (mid, RAW_N) if SFT_HALF == 0 else (0, mid)

stage_ds = shuffled.select(range(*SFT_ROWS))

print(f"[data] {DATA_DIR} train: {RAW_N} rows")
print(f"[data] shuffle(seed={DATA_SEED}) -> SFT half = rows {SFT_ROWS}, "
      f"{DPO_ROWS} reserved for DPO")
print(f"[data] sampled {len(stage_ds)} pairs for this stage")